# Enterprise Sales Forecasting Platform

**Objective:** Demonstrate a production-style end-to-end forecasting system with model comparison, automated selection, API serving, and dashboard-ready outputs.

## Problem Statement

Forecast the next 8 weeks of sales for each state using historical weekly sales data. The system compares multiple forecasting algorithms, selects the best model automatically, and serves predictions through a REST API for downstream dashboards and apps.

## System Architecture

Streamlit Dashboard  →  FastAPI Backend  →  Prediction Service  →  Forecasting Models  →  Processed Dataset

- Modular OOP architecture with clear separation of data, modeling, service, and API layers.
- Production-style backend design enabling reusable forecasting code and authenticated deployment paths.
- The notebook is a demo layer on top of the implemented project modules.

## Dataset Overview

Load the processed dataset and inspect its structure. This dataset already contains the engineered weekly features used by the forecasting models.

In [2]:
import pandas as pd
from pathlib import Path

processed_path = Path('data') / 'processed' / 'processed_timeseries.csv'
processed_df = pd.read_csv(processed_path, parse_dates=['Date', 'week_start'])
print('Dataset shape:', processed_df.shape)
print('Columns:', processed_df.columns.tolist())
processed_df.head(8)

Dataset shape: (6794, 14)
Columns: ['State', 'Date', 'Category', 'Total', 'week_start', 'lag_1', 'lag_7', 'lag_30', 'rolling_mean_7', 'rolling_std_7', 'month', 'week_of_year', 'quarter', 'is_holiday']


,State,Date,Category,Total,week_start,lag_1,lag_7,lag_30,rolling_mean_7,rolling_std_7,month,week_of_year,quarter,is_holiday
0,Alabama,2020-05-03,Beverages,128066131,2020-05-03,118114595.0,132632134.0,129106730.0,1.218652e+08,5.643463e+06,5,18,2,False
1,Alabama,2020-05-10,Beverages,127906087,2020-05-10,128066131.0,127145065.0,123782286.0,1.212130e+08,4.511690e+06,5,19,2,False
2,Alabama,2020-05-17,Beverages,130340051,2020-05-17,127906087.0,116657084.0,116218909.0,1.213217e+08,4.660055e+06,5,20,2,False
3,Alabama,2020-05-24,Beverages,134708619,2020-05-24,130340051.0,123259529.0,109968011.0,1.232764e+08,5.138641e+06,5,21,2,False
4,Alabama,2020-05-31,Beverages,129887877,2020-05-31,134708619.0,117506179.0,112189104.0,1.249120e+08,6.511631e+06,5,22,2,False
5,Alabama,2020-06-07,Beverages,132304541,2020-06-07,129887877.0,117742073.0,110932913.0,1.266808e+08,5.913937e+06,6,23,2,False
6,Alabama,2020-06-14,Beverages,131401893,2020-06-14,132304541.0,118114595.0,109056410.0,1.287611e+08,4.873452e+06,6,24,2,False
7,Alabama,2020-06-21,Beverages,133712552,2020-06-21,131401893.0,128066131.0,113040422.0,1.306593e+08,2.225083e+06,6,25,2,False


### Feature Engineering Demonstration

The project builds lag features, rolling statistics, date features, and a holiday indicator for weekly forecasting.

In [3]:
feature_columns = ['lag_1', 'lag_7', 'lag_30', 'rolling_mean_7', 'rolling_std_7', 'month', 'week_of_year', 'quarter', 'is_holiday']
processed_df[['State', 'Date', 'Total'] + feature_columns].head(10)

,State,Date,Total,lag_1,lag_7,lag_30,rolling_mean_7,rolling_std_7,month,week_of_year,quarter,is_holiday
0,Alabama,2020-05-03,128066131,118114595.0,132632134.0,129106730.0,1.218652e+08,5.643463e+06,5,18,2,False
1,Alabama,2020-05-10,127906087,128066131.0,127145065.0,123782286.0,1.212130e+08,4.511690e+06,5,19,2,False
2,Alabama,2020-05-17,130340051,127906087.0,116657084.0,116218909.0,1.213217e+08,4.660055e+06,5,20,2,False
3,Alabama,2020-05-24,134708619,130340051.0,123259529.0,109968011.0,1.232764e+08,5.138641e+06,5,21,2,False
4,Alabama,2020-05-31,129887877,134708619.0,117506179.0,112189104.0,1.249120e+08,6.511631e+06,5,22,2,False
5,Alabama,2020-06-07,132304541,129887877.0,117742073.0,110932913.0,1.266808e+08,5.913937e+06,6,23,2,False
6,Alabama,2020-06-14,131401893,132304541.0,118114595.0,109056410.0,1.287611e+08,4.873452e+06,6,24,2,False
7,Alabama,2020-06-21,133712552,131401893.0,128066131.0,113040422.0,1.306593e+08,2.225083e+06,6,25,2,False
8,Alabama,2020-06-28,132320128,133712552.0,127906087.0,109574036.0,1.314659e+08,2.161350e+06,6,26,2,False
9,Alabama,2020-07-05,138414683,132320128.0,130340051.0,108083724.0,1.320965e+08,1.602395e+06,7,27,3,False


## Model Benchmarking

Load the persisted model registry that records evaluation metrics for each supported forecasting model and highlights the automatic best-model choice.

In [4]:
import json
import pandas as pd
import plotly.express as px
from pathlib import Path

registry_path = Path('trained_models') / 'model_registry.json'
with registry_path.open('r', encoding='utf-8') as f:
    registry = json.load(f)

records = []
for state, entry in registry.items():
    models = entry.get('models', entry.get('all_models', {}))
    for model_name, metrics in models.items():
        if 'error' in metrics:
            records.append({'State': state, 'Model': model_name, 'RMSE': None, 'MAE': None, 'MAPE': None, 'Status': 'Failed'})
        else:
            records.append({
                'State': state,
                'Model': model_name,
                'RMSE': metrics.get('rmse'),
                'MAE': metrics.get('mae'),
                'MAPE': metrics.get('mape'),
                'Status': 'Best Model' if model_name == entry.get('best_model') else 'Candidate',
            })
benchmark_df = pd.DataFrame(records)
benchmark_df.head(12)

,State,Model,RMSE,MAE,MAPE,Status
0,Alabama,SARIMA,3.152502e+07,2.938477e+07,0.150030,Candidate
1,Alabama,Prophet,5.700278e+07,5.627395e+07,0.283992,Candidate
2,Alabama,XGBoost,9.723140e+06,7.888452e+06,0.040187,Best Model
3,Alabama,LSTM,1.861663e+07,1.583014e+07,0.081934,Candidate
4,Arizona,SARIMA,1.624852e+07,1.345038e+07,0.066024,Candidate
5,Arizona,Prophet,4.567475e+07,4.487636e+07,0.215883,Candidate
6,Arizona,XGBoost,1.090555e+07,9.325514e+06,0.043852,Best Model
7,Arizona,LSTM,1.579172e+07,1.360395e+07,0.064678,Candidate
8,Arkansas,SARIMA,1.317432e+07,1.188380e+07,0.114904,Candidate
9,Arkansas,Prophet,2.779171e+07,2.738959e+07,0.260743,Candidate


In [5]:
summary = benchmark_df.groupby('Model')[['RMSE', 'MAE', 'MAPE']].mean().reset_index()
fig = px.bar(summary, x='Model', y='MAPE', title='Average MAPE by Model', text='MAPE')
fig.update_traces(texttemplate='%{text:.4f}', textposition='outside')
fig.update_layout(yaxis_title='MAPE', xaxis_title='Model', template='plotly_white', margin=dict(l=20, r=20, t=50, b=20))
fig

## Forecasting Demonstration

Generate a sample forecast for Texas using the production-ready `PredictionService`. The response shows the selected model and next 8 predicted values.

In [6]:
from app.services.prediction_service import PredictionService
import plotly.express as px

service = PredictionService()
response = service.predict('Texas', forecast_periods=8)
forecast_df = pd.DataFrame({'Week': range(1, len(response['predictions']) + 1), 'Prediction': response['predictions']})
print('Selected model:', response['model_used'])
print('Forecast periods:', response['forecast_periods'])
forecast_df

s:\Forecasting System\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-07 11:38:27,002 - ModelRegistry - INFO - Loaded existing registry from S:\Forecasting System\trained_models\model_registry.json
2026-05-07 11:38:27,175 - PredictionService - INFO - Generating forecast for state=Texas periods=8 model_override=None
2026-05-07 11:38:27,176 - XGBoostForecaster - INFO - Loading XGBoost model from S:\Forecasting System\trained_models\xgboost_texas.pkl
2026-05-07 11:38:27,183 - XGBoostForecaster - INFO - XGBoost model loaded successfully
2026-05-07 11:38:27,184 - PredictionService - INFO - Loaded XGBoost model from S:\Forecasting System\trained_models\xgboost_texas.pkl
2026-05-07 11:38:27,211 - XGBoostForecaster - INFO - Generating XGBoost predictions
2026-05-07 11:38:27,216 - XGBoostForecaster - INFO - XG

Selected model: XGBoost
Forecast periods: 8


,Week,Prediction
0,1,954009216.0
1,2,954009216.0
2,3,948425984.0
3,4,951750400.0
4,5,944821888.0
5,6,950245376.0
6,7,956244736.0
7,8,952526656.0


In [7]:
fig = px.line(forecast_df, x='Week', y='Prediction', title='Texas 8-Week Sales Forecast', markers=True)
fig.update_layout(xaxis_title='Forecast Week', yaxis_title='Predicted Sales', template='plotly_white', margin=dict(l=20, r=20, t=50, b=20))
fig

## FastAPI Demonstration

The system exposes the following endpoints via the production API: 
- `POST /forecast`
- `GET /health`

Example request payload for the forecast endpoint:
```json
{
  "state": "Texas",
  "forecast_periods": 8
}
```
A typical response payload includes the selected model and forecast values, and the API is driven by the same `PredictionService` imported above.

## Dashboard Showcase

The Streamlit dashboard provides interactive forecasting controls, model comparison overlays, and benchmark visualizations. It is the presentation layer built on top of the backend and API.

## Key Achievements

- Production-ready modular forecasting architecture
- Multiple model comparison with SARIMA, Prophet, XGBoost, and LSTM
- Automatic best-model selection and registry persistence
- REST API serving forecasts via FastAPI
- Interactive dashboard-ready outputs for business users

## Future Improvements

- Deploy the system to cloud infrastructure with CI/CD
- Add real-time retraining pipelines and model drift monitoring
- Optimize hyperparameters for each model class
- Extend forecasts with prediction intervals and risk-based alerting